In [1]:
pip install requests pandas vaderSentiment yfinance

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.9 MB/s eta 0:00:01
   ------------------------------- -------- 1.3/1.7 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 2.7 MB/s eta 0:00:00

   ---------------------------------------- 0/7 [peewee]
   ---------------------------------------- 0/7 [peewee]
   ----------- ---------------------------- 2/7 [websockets]
   ----------- ---------------------------- 2/7 [websockets]
   ----------- ---------------------------- 2/7 [websockets]
  Attempting uninstall: cffi
   ----------- ---------------------------- 2/7 [websockets]
    Found existing installation: cffi 1.17.1
   ----------- ---------------------------- 2/7 [websockets]
    Uninstalling cffi-1.17.1:
   ----------- ---------------------------- 2/7 [websockets]
   ----------------- ---------------------- 3/7 [cffi]
   ----------------- ---------------------- 3/7 [cffi]
   -----

  You can safely remove it manually.


In [2]:
import pandas
import requests
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
print("All libraries are ready!")

All libraries are ready!


In [3]:
import requests
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. API perparation
API_KEY = '34d43dd677e5415893d92e3eca2f13ed'
topic = 'Bitcoin'
url = f'https://newsapi.org/v2/everything?q={topic}&language=en&apiKey={API_KEY}'

# 2. Data Extractiopn
response = requests.get(url)
data = response.json()
articles = data.get('articles', [])

# 3. Feeling Analyzer 
analyzer = SentimentIntensityAnalyzer()
results = []

for art in articles:
    text = art['title'] + " " + (art['description'] if art['description'] else "")
    score = analyzer.polarity_scores(text)['compound']
    results.append({
        'Date': art['publishedAt'],
        'Source': art['source']['name'],
        'Title': art['title'],
        'Sentiment_Score': score
    })

# 4. Save the data into DataFrame
df = pd.DataFrame(results)

                   Date       Source  \
0  2026-04-13T21:25:17Z  Gizmodo.com   
1  2026-04-08T16:40:08Z  Gizmodo.com   
2  2026-04-13T20:08:38Z     BBC News   
3  2026-04-16T21:30:44Z  Gizmodo.com   
4  2026-04-13T09:00:08Z  Gizmodo.com   

                                               Title  Sentiment_Score  
0  Popular Musician Loses Life Savings Through Ma...          -0.4939  
1  The New York Times Claims It Finally Unmasked ...           0.5267  
2  Lib Dems call for inquiry into Farage Bitcoin ...           0.0000  
3  Hollywood’s First Big Budget AI-Generated Movi...           0.0000  
4  Crypto Investor at Center of Trump Corruption ...          -0.0258  


In [4]:
df.head()

,Date,Source,Title,Sentiment_Score
0,2026-04-13T21:25:17Z,Gizmodo.com,Popular Musician Loses Life Savings Through Ma...,-0.4939
1,2026-04-08T16:40:08Z,Gizmodo.com,The New York Times Claims It Finally Unmasked ...,0.5267
2,2026-04-13T20:08:38Z,BBC News,Lib Dems call for inquiry into Farage Bitcoin ...,0.0000
3,2026-04-16T21:30:44Z,Gizmodo.com,Hollywood’s First Big Budget AI-Generated Movi...,0.0000
4,2026-04-13T09:00:08Z,Gizmodo.com,Crypto Investor at Center of Trump Corruption ...,-0.0258


In [8]:
import yfinance as yf

# 1.Convert the date column in the news
df['Date'] = pd.to_datetime(df['Date']).dt.date

# 2. Bitcoin price withdrawal
btc_data = yf.download('BTC-USD', start='2026-03-25', end='2026-04-28')

# If the columns are two levels, we will make them one level
if btc_data.columns.nlevels > 1:
    btc_data.columns = btc_data.columns.get_level_values(0)

btc_data.reset_index(inplace=True)
btc_data['Date'] = btc_data['Date'].dt.date

# 3. Merge news data with price data
# We will select Date and Close only from btc_data
final_mining_df = pd.merge(df, btc_data[['Date', 'Close']], on='Date', how='inner')

print("Success! Combined Data:")
final_mining_df.head()

[*********************100%***********************]  1 of 1 completed

Success! Combined Data:


,Date,Source,Title,Sentiment_Score,Close
0,2026-04-13,Gizmodo.com,Popular Musician Loses Life Savings Through Ma...,-0.4939,74484.640625
1,2026-04-08,Gizmodo.com,The New York Times Claims It Finally Unmasked ...,0.5267,71123.359375
2,2026-04-13,BBC News,Lib Dems call for inquiry into Farage Bitcoin ...,0.0000,74484.640625
3,2026-04-16,Gizmodo.com,Hollywood’s First Big Budget AI-Generated Movi...,0.0000,75152.132812
4,2026-04-13,Gizmodo.com,Crypto Investor at Center of Trump Corruption ...,-0.0258,74484.640625


In [12]:
# Collect data by day and calculate average sentiment and closing price
daily_mining_df = final_mining_df.groupby('Date').agg({
    'Sentiment_Score': 'mean',
    'Close': 'first' # The price is fixed per day, so it will not make a difference mean or first
}).reset_index()

# Add a column showing the price change from the previous day (Percentage Change)
daily_mining_df['Price_Diff'] = daily_mining_df['Close'].pct_change() * 100

daily_mining_df.head()

,Date,Sentiment_Score,Close,Price_Diff
0,2026-03-27,0.300725,66338.375000,NaN
1,2026-03-28,0.226300,66319.695312,-0.028158
2,2026-03-29,-0.911800,65954.921875,-0.550023
3,2026-03-30,-0.249160,66691.445312,1.116707
4,2026-03-31,-0.312440,68233.312500,2.311941


In [13]:
daily_mining_df['Price_Diff'] = daily_mining_df['Price_Diff'].fillna(0)

In [14]:
daily_mining_df.head()

,Date,Sentiment_Score,Close,Price_Diff
0,2026-03-27,0.300725,66338.375000,0.000000
1,2026-03-28,0.226300,66319.695312,-0.028158
2,2026-03-29,-0.911800,65954.921875,-0.550023
3,2026-03-30,-0.249160,66691.445312,1.116707
4,2026-03-31,-0.312440,68233.312500,2.311941


In [15]:
!pip install pymysql sqlalchemy

In [17]:
import pymysql
import pandas as pd
from sqlalchemy import create_engine

username = "root"
password = "###############"
host = "localhost"
port = "3306"
database = "crypto_sentiment_analysis"

connection = pymysql.connect(host=host, user=username, password=password)
cursor = connection.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {database};")
connection.close()

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")


daily_mining_df.to_sql('daily_mining_data', engine, if_exists='replace', index=False)
pd.read_sql("SELECT * FROM daily_mining_data LIMIT 5;", engine)

,Date,Sentiment_Score,Close,Price_Diff
0,2026-03-27,0.300725,66338.375000,0.000000
1,2026-03-28,0.226300,66319.695312,-0.028158
2,2026-03-29,-0.911800,65954.921875,-0.550023
3,2026-03-30,-0.249160,66691.445312,1.116707
4,2026-03-31,-0.312440,68233.312500,2.311941
